# 03. Relación entre mantenimiento y fallas

Vamos a responder dos preguntas:

1. **¿Qué registros de mantenimiento están asociados a una falla?**
2. **¿Qué riesgo implica usarlos como variables de entrada?**

Relacionaremos los archivos por máquina, componente y fecha-hora. También revisaremos los casos sin coincidencia y un ejemplo de qué reemplazos se conocían antes y después de una falla.

El análisis de eventos usará **enero–agosto de 2015**. Para consultar el último reemplazo anterior a una fecha conservaremos además el historial de mantenimiento de 2014. No estudiaremos patrones de validación ni de prueba.

## 1. Preparar las tablas

El componente se llama `comp` en mantenimiento y `failure` en fallas. En las copias de trabajo lo llamaremos `componente` para facilitar el cruce. Los nombres y valores de los CSV originales se mantienen.

Separaremos el mantenimiento de entrenamiento del historial anterior a septiembre. Ese historial permite conocer reemplazos previos a enero sin contarlos como eventos de entrenamiento.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

# Encontrar Datos desde la raíz del proyecto o desde Preprocesamiento.
carpeta_actual = Path.cwd()
if (carpeta_actual / "Datos").is_dir():
    ruta_repo = carpeta_actual
elif (carpeta_actual.parent / "Datos").is_dir():
    ruta_repo = carpeta_actual.parent
else:
    raise FileNotFoundError("No se encontró la carpeta Datos.")

ruta_datos = ruta_repo / "Datos"
inicio = pd.Timestamp("2015-01-01")
fin = pd.Timestamp("2015-09-01")
componentes = ["comp1", "comp2", "comp3", "comp4"]
pd.set_option("display.max_columns", 12)
pd.set_option("display.max_rows", 30)

# Convertir las fechas al leer y unificar el nombre del componente en memoria.
mantenimiento_original = pd.read_csv(ruta_datos / "PdM_maint.csv", parse_dates=["datetime"])
fallas_original = pd.read_csv(ruta_datos / "PdM_failures.csv", parse_dates=["datetime"])
mantenimiento_original = mantenimiento_original.rename(columns={"comp": "componente"})
fallas_original = fallas_original.rename(columns={"failure": "componente"})

# El historial incluye 2014, pero nunca septiembre o fechas posteriores.
historial_mantenimiento = mantenimiento_original.loc[
    mantenimiento_original["datetime"] < fin
].copy()
mantenimiento = historial_mantenimiento.loc[
    historial_mantenimiento["datetime"] >= inicio
].copy()
fallas = fallas_original.loc[
    (fallas_original["datetime"] >= inicio) & (fallas_original["datetime"] < fin)
].copy()

# Descartar las tablas originales de la sesión para trabajar solo con los periodos elegidos.
del mantenimiento_original, fallas_original
claves = ["machineID", "datetime", "componente"]

# Estas comprobaciones detienen el análisis si el cruce no tendría una clave única.
assert not mantenimiento.duplicated(claves).any()
assert not fallas.duplicated(claves).any()
assert mantenimiento["datetime"].between(inicio, fin, inclusive="left").all()
assert fallas["datetime"].between(inicio, fin, inclusive="left").all()

resumen = pd.DataFrame({
    "tabla": ["Mantenimiento de entrenamiento", "Fallas de entrenamiento", "Mantenimientos anteriores a enero"],
    "registros": [len(mantenimiento), len(fallas), int((historial_mantenimiento["datetime"] < inicio).sum())],
})
display(resumen)

,tabla,registros
0,Mantenimiento de entrenamiento,1916
1,Fallas de entrenamiento,522
2,Mantenimientos anteriores a enero,400


Quedaron **1,916 registros de mantenimiento y 522 registros de falla** de enero–agosto. Además, conservamos **400 mantenimientos anteriores a enero** como historia previa. Las comprobaciones confirman que no hay claves repetidas en los dos archivos que vamos a cruzar.

## 2. Identificar coincidencias exactas

Un mantenimiento se considera **asociado a una falla registrada** cuando coinciden las tres claves: máquina, componente y hora. Es la evidencia disponible para identificar reemplazos correctivos; los archivos no tienen una columna adicional que declare el motivo o el momento en que el registro estuvo disponible.

Usaremos un cruce externo para conservar tanto los eventos coincidentes como los que aparecen en un solo archivo. No basta con cruzar por máquina y hora: podrían ser componentes diferentes.

In [2]:
# outer conserva todos los registros. indicator informa de qué tabla procede cada uno.
# one_to_one exige que una clave represente como máximo una fila en cada archivo.
cruce = mantenimiento.merge(
    fallas,
    on=claves,
    how="outer",
    indicator=True,
    validate="one_to_one",
)

# Asignar una descripción sin asumir que todo reemplazo no coincidente fue programado.
etiquetas_cruce = {
    "both": "Mantenimiento asociado a falla",
    "left_only": "Mantenimiento sin falla coincidente",
    "right_only": "Falla sin mantenimiento coincidente",
}
cruce["relacion"] = cruce["_merge"].astype(str).map(etiquetas_cruce)
asociados = cruce.loc[cruce["_merge"] == "both", claves].copy()
solo_mantenimiento = cruce.loc[cruce["_merge"] == "left_only", claves].copy()
solo_falla = cruce.loc[cruce["_merge"] == "right_only", claves].copy()

# Comprobar que cada archivo original queda completamente explicado por sus dos grupos.
assert len(asociados) + len(solo_mantenimiento) == len(mantenimiento)
assert len(asociados) + len(solo_falla) == len(fallas)

resumen_cruce = cruce["relacion"].value_counts().rename("registros").to_frame()
display(resumen_cruce)
print("Mantenimientos asociados a falla (% del mantenimiento):", round(100 * len(asociados) / len(mantenimiento), 2))
print("Fallas con mantenimiento coincidente (% de las fallas):", round(100 * len(asociados) / len(fallas), 2))

,registros
relacion,
Mantenimiento sin falla coincidente,1412
Mantenimiento asociado a falla,504
Falla sin mantenimiento coincidente,18


Mantenimientos asociados a falla (% del mantenimiento): 26.3
Fallas con mantenimiento coincidente (% de las fallas): 96.55


De los **1,916 mantenimientos de entrenamiento, 504 (26.30%) coinciden con una falla** de la misma máquina y componente en la misma hora. Son los reemplazos que podemos identificar como asociados a una falla registrada.

Los otros **1,412 mantenimientos (73.70%) no tienen una falla coincidente**. Son compatibles con reemplazos programados u otros motivos, pero la ausencia de coincidencia no prueba por sí sola que todos fueran preventivos.

De las **522 fallas**, **504 (96.55%)** tienen mantenimiento coincidente y **18 (3.45%)** no. Por ello, no sería correcto asumir que ambos archivos se corresponden perfectamente.

## 3. Revisar los resultados por componente y ver ejemplos

Contaremos cada relación por componente y mostraremos ejemplos de coincidencias. Mantendremos separados los componentes que fallan en una misma hora para no perder información.

In [3]:
# crosstab cuenta combinaciones de componente y tipo de relación.
por_componente = pd.crosstab(cruce["componente"], cruce["relacion"]).reindex(componentes, fill_value=0)
por_componente = por_componente.reindex(columns=list(etiquetas_cruce.values()), fill_value=0)

# Calcular el porcentaje dentro de los mantenimientos de cada componente, no sobre todas las fallas.
por_componente["total_mantenimientos"] = (
    por_componente["Mantenimiento asociado a falla"]
    + por_componente["Mantenimiento sin falla coincidente"]
)
por_componente["porcentaje_asociado_a_falla"] = (
    100 * por_componente["Mantenimiento asociado a falla"] / por_componente["total_mantenimientos"]
)
display(por_componente.round(2))

# Elegir los primeros ejemplos cronológicos, sin escogerlos por su resultado.
ejemplos = asociados.sort_values(["datetime", "machineID", "componente"]).head(8)
display(ejemplos)

relacion,Mantenimiento asociado a falla,Mantenimiento sin falla coincidente,Falla sin mantenimiento coincidente,total_mantenimientos,porcentaje_asociado_a_falla
componente,,,,,
comp1,132,332,9,464,28.45
comp2,165,345,3,510,32.35
comp3,88,381,3,469,18.76
comp4,119,354,3,473,25.16


,machineID,datetime,componente
435,24,2015-01-02 06:00:00,comp1
1404,73,2015-01-02 06:00:00,comp3
1552,81,2015-01-02 06:00:00,comp1
364,20,2015-01-04 06:00:00,comp2
693,37,2015-01-04 06:00:00,comp4
751,40,2015-01-04 06:00:00,comp3
1,1,2015-01-05 06:00:00,comp4
418,23,2015-01-05 06:00:00,comp2


Las coincidencias se reparten en **132 registros de comp1, 165 de comp2, 88 de comp3 y 119 de comp4**. Los porcentajes asociados a fallas se calculan dentro del mantenimiento de cada componente, de modo que el denominador queda explícito.

Los ejemplos muestran el tipo de coincidencia usada: no estamos uniendo un reemplazo de un componente con la falla de otro. Tampoco estamos eliminando eventos simultáneos.

## 4. Investigar las 18 fallas sin mantenimiento coincidente

Primero veremos si se concentran en alguna fecha. Luego consultaremos el reemplazo anterior y el siguiente del **mismo componente y máquina**.

La búsqueda posterior se limita al historial disponible antes de septiembre. Es una revisión retrospectiva de calidad, **no información para un predictor**. Una fecha cercana no demuestra que el reemplazo responda a esa falla. No ajustaremos las horas ni completaremos registros automáticamente.

In [4]:
# Mostrar todas las excepciones: son pocas y conviene revisarlas sin ocultarlas.
solo_falla = solo_falla.sort_values(["datetime", "machineID", "componente"])
display(solo_falla)
display(solo_falla.groupby("datetime").size().rename("fallas_sin_coincidencia").to_frame())

# Consultar el mantenimiento anterior y posterior para cada excepción.
# NaT indica que no hay un evento en esa dirección dentro del historial revisado.
detalle_excepciones = []
for evento in solo_falla.itertuples(index=False):
    historial = historial_mantenimiento.loc[
        (historial_mantenimiento["machineID"] == evento.machineID)
        & (historial_mantenimiento["componente"] == evento.componente),
        "datetime",
    ]

    anteriores = historial[historial < evento.datetime]
    posteriores = historial[historial > evento.datetime]
    anterior = anteriores.max()
    posterior = posteriores.min()
    horas_hasta_posterior = (posterior - evento.datetime).total_seconds() / 3600 if pd.notna(posterior) else np.nan

    detalle_excepciones.append({
        "machineID": evento.machineID,
        "componente": evento.componente,
        "fecha_falla": evento.datetime,
        "reemplazo_anterior": anterior,
        "reemplazo_posterior": posterior,
        "horas_hasta_posterior": horas_hasta_posterior,
    })

detalle_excepciones = pd.DataFrame(detalle_excepciones)
display(detalle_excepciones)

# Usar tolerancias solo como diagnóstico: no reclasificar estos casos como coincidencias.
revision_tolerancias = []
for horas in [3, 6, 24]:
    dentro = detalle_excepciones["horas_hasta_posterior"].between(0, horas, inclusive="both")
    revision_tolerancias.append({"horas_posteriores_revisadas": horas, "fallas_con_reemplazo_posterior": int(dentro.sum())})
display(pd.DataFrame(revision_tolerancias))

,machineID,datetime,componente
285,16,2015-01-02 03:00:00,comp1
286,16,2015-01-02 03:00:00,comp3
305,17,2015-01-02 03:00:00,comp4
397,22,2015-01-02 03:00:00,comp1
650,35,2015-01-02 03:00:00,comp1
853,45,2015-01-02 03:00:00,comp1
963,51,2015-01-02 03:00:00,comp1
1055,56,2015-01-02 03:00:00,comp3
1095,58,2015-01-02 03:00:00,comp2
1114,59,2015-01-02 03:00:00,comp1


,fallas_sin_coincidencia
datetime,
2015-01-02 03:00:00,18


,machineID,componente,fecha_falla,reemplazo_anterior,reemplazo_posterior,horas_hasta_posterior
0,16,comp1,2015-01-02 03:00:00,2014-11-13 06:00:00,2015-01-17 06:00:00,363.0
1,16,comp3,2015-01-02 03:00:00,2014-10-29 06:00:00,2015-03-03 06:00:00,1443.0
2,17,comp4,2015-01-02 03:00:00,2015-01-01 06:00:00,2015-01-31 06:00:00,699.0
3,22,comp1,2015-01-02 03:00:00,2014-12-28 06:00:00,2015-01-22 06:00:00,483.0
4,35,comp1,2015-01-02 03:00:00,2014-12-13 06:00:00,2015-02-20 06:00:00,1179.0
5,45,comp1,2015-01-02 03:00:00,2014-12-28 06:00:00,2015-02-06 06:00:00,843.0
6,51,comp1,2015-01-02 03:00:00,2015-01-01 06:00:00,2015-01-16 06:00:00,339.0
7,56,comp3,2015-01-02 03:00:00,2014-12-13 06:00:00,2015-01-14 06:00:00,291.0
8,58,comp2,2015-01-02 03:00:00,2014-08-30 06:00:00,2015-01-31 06:00:00,699.0
9,59,comp1,2015-01-02 03:00:00,2014-12-28 06:00:00,2015-01-20 06:00:00,435.0


,horas_posteriores_revisadas,fallas_con_reemplazo_posterior
0,3,0
1,6,0
2,24,0


Las **18 fallas sin coincidencia exacta están registradas el 2 de enero de 2015 a las 03:00**. Ninguna tiene un reemplazo del mismo componente en las **3, 6 o 24 horas posteriores** dentro del historial revisado. Por tanto, esos casos no se explican simplemente admitiendo un retraso corto de registro.

La concentración temporal merece quedar documentada, pero no permite determinar por sí sola si existe un problema de carga, una particularidad de la simulación u otra causa. Los reemplazos anteriores tampoco deben atribuirse como consecuencia de una falla posterior. **Conservamos los registros y no cambiamos sus fechas.**

## 5. Mostrar qué mantenimiento estaba disponible al predecir

El riesgo no consiste en que todo mantenimiento sea inutilizable. Consiste en **adelantar información** que solo aparece después de la hora de predicción.

Tomaremos una coincidencia con al menos 72 horas desde el inicio de la telemetría y consultaremos el último reemplazo de su componente dos horas antes, una hora antes, a la hora de falla y una hora después.

Para este ejemplo usaremos la regla conservadora **fecha de reemplazo < hora de consulta**. Los registros de la misma hora se dejan para la siguiente, porque no contamos con un orden dentro de esa hora ni con la hora de carga del registro. En producción también habría que respetar los retrasos reales de captura.

La fecha de falla se usa aquí solo para explicar la secuencia; no se entrega como entrada al modelo.

In [5]:
# Elegir un ejemplo de forma reproducible: primera coincidencia después de 72 horas de cobertura.
elegibles = asociados.loc[asociados["datetime"] >= pd.Timestamp("2015-01-04 06:00:00")]
ejemplo = elegibles.sort_values(["datetime", "machineID", "componente"]).iloc[0]
fecha_evento = ejemplo["datetime"]

historial_ejemplo = historial_mantenimiento.loc[
    (historial_mantenimiento["machineID"] == ejemplo["machineID"])
    & (historial_mantenimiento["componente"] == ejemplo["componente"]),
    "datetime",
].sort_values()

# Para cada hora, buscar solo hacia atrás. No rellenar desde un reemplazo futuro.
secuencia = []
for desplazamiento in [-2, -1, 0, 1]:
    hora_consulta = fecha_evento + pd.Timedelta(hours=desplazamiento)
    conocidos = historial_ejemplo[historial_ejemplo < hora_consulta]
    ultimo = conocidos.max()
    horas_desde = (hora_consulta - ultimo).total_seconds() / 3600 if pd.notna(ultimo) else np.nan

    secuencia.append({
        "hora_consulta": hora_consulta,
        "horas_respecto_falla": desplazamiento,
        "ultimo_reemplazo_anterior": ultimo,
        "horas_desde_reemplazo": horas_desde,
        "reemplazo_de_esta_falla_ya_disponible": fecha_evento < hora_consulta,
    })

print("Ejemplo: máquina", ejemplo["machineID"], "-", ejemplo["componente"])
display(pd.DataFrame(secuencia))

Ejemplo: máquina 20 - comp2


,hora_consulta,horas_respecto_falla,ultimo_reemplazo_anterior,horas_desde_reemplazo,reemplazo_de_esta_falla_ya_disponible
0,2015-01-04 04:00:00,-2,2014-06-01 06:00:00,5206.0,False
1,2015-01-04 05:00:00,-1,2014-06-01 06:00:00,5207.0,False
2,2015-01-04 06:00:00,0,2014-06-01 06:00:00,5208.0,False
3,2015-01-04 07:00:00,1,2015-01-04 06:00:00,1.0,True


Antes de la falla, su reemplazo **todavía no está disponible**. Si pegáramos ese reemplazo a horas anteriores mediante un cruce hacia adelante, estaríamos dando al modelo una pista del evento que queremos anticipar.

Con la regla conservadora del ejemplo, la hora de falla tampoco utiliza el reemplazo de esa misma hora. Una hora después, el reemplazo ya puede formar parte del historial: es información pasada para nuevas predicciones. No debemos eliminar para siempre los mantenimientos asociados a fallas, porque sí pueden servir como historia después de haber ocurrido y estar registrados.

La etiqueta futura será sobre **(t, t + 24 h]**. Una falla exactamente en `t` queda fuera de ese intervalo. Por eso es importante no confundir predecir una falla futura con reconocer una falla que ya ocurrió.

## 6. Respuestas y reglas para los siguientes pasos

### ¿Qué registros de mantenimiento son consecuencia de una falla?

En entrenamiento identificamos **504 registros** que coinciden en máquina, componente y hora con una falla. Son la evidencia de reemplazos asociados a fallas que ofrecen estos archivos. No podemos demostrar el orden dentro de esa misma hora ni el momento de captura, porque no están documentados.

Los **1,412 registros sin coincidencia** no se clasificarán automáticamente como preventivos. Las **18 fallas sin reemplazo coincidente** quedarán documentadas como excepciones; no se inventarán reemplazos para completarlas.

### ¿Qué riesgo implica usarlos como variable de entrada?

Si usamos el mantenimiento que responde a una falla para predecir esa misma falla desde horas anteriores, introducimos **fuga de información**: el modelo recibe una consecuencia del evento antes de que pudiera conocerla. Sus métricas podrían parecer muy buenas y no representar lo que pasaría al usarlo en tiempo real.

| Operación | Riesgo o regla que seguiremos |
|---|---|
| Buscar el próximo reemplazo o las horas hasta él | Usa información futura; no se usará como predictor. |
| Cruzar con el evento más cercano sin controlar la dirección | Puede seleccionar un reemplazo posterior; buscar solo hacia atrás. |
| Rellenar hacia atrás con `bfill` | Puede trasladar un reemplazo futuro a horas anteriores; evitarlo en estas variables. |
| Calcular ventanas centradas o incluir el día completo | Puede incorporar horas posteriores a la predicción; usar ventanas históricas. |
| Horas desde el último reemplazo | Es utilizable si el reemplazo ya ocurrió y estaba disponible; aplicar el corte temporal por máquina y componente. |
| No conocer un reemplazo anterior | Mantener la ausencia de historia y definir su tratamiento después; no sustituirla con el primer evento futuro. |
| Mantenimientos de 2014 | Pueden aportar historia previa; no cuentan como eventos de 2015 ni garantizan que el registro sea exhaustivo. |
| Reemplazos registrados en la misma hora de consulta | Adoptar por ahora la regla estricta `< t`; documentar cualquier cambio si se conoce el orden o la disponibilidad. |

Las coincidencias con fallas se usan **para auditar los datos**, no como una columna predictiva. Las variables definitivas se construirán más adelante y deberán respetar tanto el tiempo de los eventos como su disponibilidad.

**Siguiente paso:** construir las etiquetas a 24 horas y la partición temporal, incluyendo los casos con varios componentes.